# Siding Repair: Subtopic Content Generator

Loads all siding-repair cluster stubs, drills down to one cluster and one subtopic,
retrieves RAG context from the construction books LanceDB, and generates ~200-word content.

Tune parameters in the **Config** cell below before running.

In [25]:
from pathlib import Path

LANCEDB_PATH = r"C:\Users\tfalcon\microsites\tools\programatic writing stuffs\DBA writer\lancedb_construction_books"
LANCEDB_TABLE = "construction_books"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MODEL = "gpt-4o"
TEMPERATURE = 0.7
TOP_K = 20
WORD_TARGET = 200

SIDING_REPAIR_DIR = Path("../../../apps/siding-repair/src/data/generated_content")
AGENTS_DIR = Path("../agents")
CONTENT_REVIEW_DIR = Path("../../content-review")

In [26]:
import sys, os, re
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(CONTENT_REVIEW_DIR.resolve()))

import lancedb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from agent_loader import load_agent

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embedder = SentenceTransformer(EMBEDDING_MODEL)
db = lancedb.connect(LANCEDB_PATH)
table = db.open_table(LANCEDB_TABLE)
agent = load_agent(AGENTS_DIR / "01-subtopic-writer.md")
print(f"LanceDB: {table.count_rows()} docs | Agent: {agent.name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LanceDB: 6260 docs | Agent: Subtopic Writer


In [27]:
def parse_cluster_stub(path: Path) -> dict | None:
    content = path.read_text(encoding="utf-8")
    meta_match = re.search(r'<!--\s*CLUSTER_META([\s\S]*?)-->', content)
    if not meta_match:
        return None
    meta, subtopics, in_subtopics = {}, [], False
    for line in meta_match.group(1).split('\n'):
        s = line.strip()
        if s.startswith('subtopics:'):
            in_subtopics = True; continue
        if in_subtopics:
            if s.startswith('- '): subtopics.append(s[2:].strip())
            elif s and not s.startswith('-'): in_subtopics = False
        if ':' in s and not in_subtopics:
            k, v = s.split(':', 1)
            meta[k.strip()] = v.strip()
    meta['subtopics'] = subtopics
    meta['filename'] = path.name
    return meta

stubs = [s for s in (parse_cluster_stub(f) for f in sorted(SIDING_REPAIR_DIR.glob("service_page_cluster_*.md"))) if s]
print(f"{'Slug':<55} {'Loc':<10} Subtopics")
print("-" * 75)
for s in sorted(stubs, key=lambda x: (x.get('cluster_slug',''), x.get('location',''))):
    print(f"{s.get('cluster_slug','?'):<55} {s.get('location','?'):<10} {len(s['subtopics'])}")

Slug                                                    Loc        Subtopics
---------------------------------------------------------------------------
exterior-wall-sheathing-weather-barrier-repair          portland   5
exterior-wall-sheathing-weather-barrier-repair          seattle    5
hardie-fiber-cement-siding-repair                       portland   1
hardie-fiber-cement-siding-repair                       seattle    1
lap-cedar-wood-specialty-siding-repair                  portland   4
lap-cedar-wood-specialty-siding-repair                  seattle    4
siding-flashing-water-intrusion-repair                  portland   4
siding-flashing-water-intrusion-repair                  seattle    4
siding-integration-repairs                              portland   3
siding-integration-repairs                              seattle    3
siding-rot-repair-replacement                           portland   7
siding-rot-repair-replacement                           seattle    7
siding-trim-corner-

In [28]:
TARGET_SLUG = "exterior-wall-sheathing-weather-barrier-repair"
TARGET_LOCATION = "portland"

cluster = next(s for s in stubs if s.get('cluster_slug') == TARGET_SLUG and s.get('location') == TARGET_LOCATION)
print(f"{cluster['cluster_slug']} ({cluster['location']})")
for i, t in enumerate(cluster['subtopics']):
    print(f"  [{i}] {t}")

exterior-wall-sheathing-weather-barrier-repair (portland)
  [0] Exterior Wall Sheathing Repair
  [1] Weather Barrier Repair Behind Siding
  [2] House Wrap Replacement During Siding Repair
  [3] Wall Sheathing Rot Replacement
  [4] Structural Framing Repair Behind Siding


In [29]:
TARGET_SUBTOPIC_INDEX = 0  # change to pick a different subtopic
TARGET_SUBTOPIC = cluster['subtopics'][TARGET_SUBTOPIC_INDEX]

query = f"{TARGET_SUBTOPIC} {cluster.get('service','siding repair')} Portland"
results = table.search(embedder.encode([query])[0]).limit(TOP_K).to_pandas()

print(f"Subtopic: {TARGET_SUBTOPIC}\n")
for i, row in results.iterrows():
    print(f"[{i+1}] {row.get('source','?')} p.{row.get('page','?')}  dist={row['_distance']:.4f}")
    print(f"     {str(row['text'])[:150]}...\n")

Subtopic: Exterior Wall Sheathing Repair

[1] Michael Litchfield - Renovation-Taunton Press (2013).pdf p.426  dist=0.8717
     d,
ing to the type of siding. It’s possible to drill
through wood siding and plug it later; but it’s
preferable to remove the wood siding, drill
throu...

[2] Charlie Wing - The Visual Handbook of Building and Remodeling-Taunton Press (2018).pdf p.252  dist=0.8986
     or installation over other than
•Treat both sides of siding with water repellent
solid wood sheathing and may require predrilling to
before installati...

[3] Charlie Wing - The Visual Handbook of Building and Remodeling-Taunton Press (2018).pdf p.252  dist=0.9137
     Vertical Wood Siding
W
ood changes in width and thickness with changes Recommended nailing patterns are shown on the
in moisture content. To minimize ...

[4] (For Pros By Pros) Christina Glennon - Siding, Roofing, and Trim-Taunton Press (2014).pdf p.44  dist=0.9286
     No matter where you live, the siding of your large tears with

In [30]:
location_full = "Portland, Oregon" if TARGET_LOCATION == "portland" else "Seattle, Washington"
context = "\n\n---\n\n".join(
    f"Source: {row.get('source','?')} (Page {row.get('page','N/A')})\n{row['text']}"
    for _, row in results.iterrows()
)
user_prompt = f"""Reference material from construction books:

{context}

---

Write a ~{WORD_TARGET}-word section for the following:

Subtopic: {TARGET_SUBTOPIC}
Parent page: {cluster.get('service','siding repair')} - {location_full}

Do not include the heading. Return only the section body content and the references table"""

response = client.chat.completions.create(
    model=MODEL,
    temperature=TEMPERATURE,
    messages=[
        {"role": "system", "content": agent.system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)
generated = response.choices[0].message.content
print(f"=== {TARGET_SUBTOPIC} ===\n\n{generated}")

=== Exterior Wall Sheathing Repair ===

Repairing exterior wall sheathing is crucial for maintaining the structural integrity and weather resistance of your home, especially in Portland's wet climate. Damaged sheathing can lead to moisture infiltration, compromising insulation and potentially causing rot and mold issues. When repairing, it's essential to first remove the siding carefully to access the sheathing. For wood siding, detaching it without causing damage involves prying up the boards gently and ensuring the weather-resistive barrier is not compromised^1. Upon reaching the sheathing, assess the extent of damage. If the sheathing is wood, ensure any replacement pieces match the existing material in thickness and type for a seamless repair. Additionally, using weather-resistant nails like stainless steel or hot-dipped galvanized ensures longevity and prevents corrosion^2. Once the sheathing is repaired, reinstall the siding, making sure to secure it properly and seal any joints 

In [24]:
words = len(generated.split())
sents = len(re.findall(r'[.!?]+', generated))
print(f"Words: {words} (target {WORD_TARGET}) | Sentences: {sents} | Avg: {words/max(sents,1):.1f} w/s")

Words: 225 (target 200) | Sentences: 13 | Avg: 17.3 w/s
